## DS256 - Scalable Systems for Data Science | Jan 2026
# Assignment 1: LLM Data Preprocessing Pipeline with Apache Spark

## Maximum Points: 100

### Overview
----
This assignment implements a complete **LLM Data Preprocessing Pipeline** using Apache Spark on a YARN cluster. The pipeline processes raw Common Crawl WARC data through 5 stages to produce clean, tokenized training data suitable for Large Language Model training.

### Pipeline Stages
| Stage | Name | Description |
|-------|------|-------------|
| 1 | Warc to Parquet | Convert the warc files to parquet (5%) |
| 2 | Ingestion / Filter | Load WARC parquet files and filter blacklisted domains (15%) |
| 3 | Extraction | Extract clean text from HTML using Trafilatura (15%) |
| 4 | Language ID | Filter to English-only documents using FastText (15%) |
| 5 | Deduplication | Remove near-duplicate documents using MinHash LSH (30%) |
| 6 | Tokenization | Convert text to token sequences for model training (20%) |

### Common Instructions
----
* You must ONLY edit cells and regions within those cells that allow changes. **DO NOT MODIFY** cells marked with `DO NOT MODIFY`.
* All processing should be done **within Spark** using DataFrames, RDDs, and Spark SQL.
* Ensure your virtual environment Python is accessible to all YARN worker nodes.

In [1]:
print("hello world!")

hello world!


In [1]:
# ======== DO NOT MODIFY ===========
!pip install pyspark
!pip install warcio
!pip install fasttext

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 8.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pybind11-3.0.2-py3-none-any.whl.metadata (10 kB)
Using cached pybind11-3.0.2-py3-none-any.whl (310 kB)
  Created wheel for fasttext: filename=fasttext-0.9.3-cp312-cp312-linux_x86_64.whl size=4647421 sha256=4677fde51ab2e75ec9016d2f0b132b322ed68af040aaf3d443e184b4d8d0e1f5
  Stored in directory: /root/.cache/pip/wheels/20/27/95/a7baf1b435f1cbde017cabdf1e9688526d2b0e929255a359c6
Successfully built fasttext


In [ ]:
# ======== DO NOT MODIFY ===========
import sys
import os
import subprocess
import time
import pyspark

from urllib.parse import urlparse
from pyspark.sql.types import StructType, StructField, StringType, BinaryType
from pyspark.sql import SparkSession
from google.colab import drive
from warcio.archiveiterator import ArchiveIterator

In [ ]:
# ======== DO NOT MODIFY ===========
drive.mount('/content/drive', force_remount=True)

In [ ]:
# ======== DO NOT MODIFY ===========
BASE_DIR = '/content/drive/Shareddrives/ds256-2026-public/assignment_1/small' # we will change this to large. You can test with small and medium.
BLACKLIST_DOMAINS_DIR = '/content/drive/Shareddrives/ds256-2026-public/assignment_1/blacklist_domains'
FASTTEXT_MODEL_BIN = '/content/drive/Shareddrives/ds256-2026-public/assignment_1/lid.176.bin'
PARQUET_DIR = '/content/drive/MyDrive/<name_of_output_dir>/' #TODO: Change it to a dir on your drive

In [ ]:
# ======= DO NOT MODIFY ============
spark = (
    SparkSession.builder
    .appName("LLMSpark")
    .master("local[*]")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .getOrCreate()
)

## Verify the warc files are readable

In [ ]:
# ======= DO NOT MODIFY ============
os.listdir(BASE_DIR)

In [ ]:
# ===== SCHEMA VALIDATORS FOR EACH STAGE (DO NOT MODIFY) =====

# Stage 2 Output Schema Validator
STAGE_2_SCHEMA = {
    'warc_id': 'string',
    'url': 'string',
    'date': 'string',
    'html_content': 'string'
}

# Stage 3 Output Schema Validator
STAGE_3_SCHEMA = {
    'warc_id': 'string',
    'url': 'string',
    'date': 'string',
    'extracted_text': 'string'
}

# Stage 4 Output Schema Validator (same as Stage 2)
STAGE_4_SCHEMA = {
    'warc_id': 'string',
    'url': 'string',
    'date': 'string',
    'extracted_text': 'string'
}

# Stage 5 Output Schema Validator (same as Stage 3)
STAGE_5_SCHEMA = {
    'warc_id': 'string',
    'url': 'string',
    'date': 'string',
    'extracted_text': 'string'
}

# Stage 6 Output Schema Validator
STAGE_6_SCHEMA = {
    'warc_id': 'string',
    'tokens': 'array<int>',
    'attention_mask': 'array<int>'
}

def validate_stage_schema(df, stage_num):
    """Validate DataFrame schema for a specific stage."""
    schemas = {
        2: STAGE_2_SCHEMA,
        3: STAGE_3_SCHEMA,
        4: STAGE_4_SCHEMA,
        5: STAGE_5_SCHEMA,
        6: STAGE_6_SCHEMA,
    }
    expected = schemas.get(stage_num, {})
    validate_schema(df, expected)
    print(f"✓ Stage {stage_num} schema validation passed!")

In [ ]:
# ===== SCHEMA VALIDATORS FOR EACH STAGE (DO NOT MODIFY) =====

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.storagelevel import StorageLevel

# Helper for schema validation (DO NOT MODIFY)
def validate_schema(df, expected_columns):
    """Verifies that the dataframe contains the required columns with correct types."""
    df_cols = {f.name: f.dataType.simpleString() for f in df.schema.fields}
    missing = []
    for col, type_str in expected_columns.items():
        if col not in df_cols:
            missing.append(f"{col} (missing)")
        elif type_str not in df_cols[col]:
             # Loose check to allow 'string' vs 'string (nullable = true)'
            pass
    if missing:
        raise ValueError(f"Schema Validation Failed. Issues: {missing}")
    else:
        print("Schema Validation Passed!")

---
---
## Your Code Edits Start from Here
---
## Stage 1: Convert the warc files to parquet ***(5 points)***
----

### Objective
Ingest raw WARC data from the HDFS file system (Google Drive in this Colab Notebook), parse the records to extract HTML content, and save the result as a partitioned Parquet dataset.

The function returns nothing. It just writes the parquet files to OUTPUT_DIR.

### Guidelines
1. List Files: Retrieve the list of .warc files from the BASE_DIR directory.
2. Stream each WARC file.
3. Parse records using warcio.ArchiveIterator.
4. Filter for records where rec_type is 'response' and Content-Type contains 'html'.
5. Write to Parquet: Save the DataFrame to OUTPUT_DIR in Parquet format, partitioned by the original WARC filename.

***The DataFrame should have the output schema defined below.***
### Parquet DataFrame Schema
| Column | Type | Description |
|--------|------|-------------|
| warc_id | string | Unique WARC record identifier |
| url | string | Source URL |
| date | string | Crawl timestamp |
| html_content | string | Raw HTML content |
| warc_filename | string | The name of the source WARC file (used for partitioning)

In [ ]:
# ***** INSTALL YOUR IMPORTS HERE ******
!pip install trafilatura
# ***** INSTALL YOUR IMPORTS HERE ******

In [ ]:
# ***** IMPORT YOUR PACKAGES HERE ******
import trafilatura
from pyspark import SparkFiles
import fasttext
# ***** IMPORT YOUR PACKAGES HERE ******

In [ ]:
#######################################
###!@1 START ANSWER STAGE 1
#######################################

def stage_1_warc_to_parquet():
    """
    Reads all .warc files from BASE_DIR, parses HTML response records
    using warcio, and writes the result as a Parquet dataset to PARQUET_DIR.
    The output is partitioned by the source WARC filename.
    Returns nothing — side-effect is writing Parquet files.
    """

    # Define the output schema explicitly.
    # Always define schemas manually in Spark — never rely on inference
    # when the data has nulls or mixed types, as Spark may guess wrong.
    schema = StructType([
        StructField("warc_id",       StringType(), True),
        StructField("url",           StringType(), True),
        StructField("date",          StringType(), True),
        StructField("html_content",  StringType(), True),
        StructField("warc_filename", StringType(), True),
    ])

    # List only .warc files in BASE_DIR (ignores any other files/folders).
    warc_files = [f for f in os.listdir(BASE_DIR) if f.endswith(".warc")]
    print(f"Found {len(warc_files)} WARC files: {warc_files}")

    all_rows = []

    for filename in warc_files:
        filepath = os.path.join(BASE_DIR, filename)
        print(f"  Parsing: {filename}")

        # Open in binary mode — WARC files are binary-encoded.
        with open(filepath, "rb") as f:

            # ArchiveIterator streams through the file record by record.
            # It never loads the whole WARC into memory — safe for large files.
            for record in ArchiveIterator(f):

                # WARC files contain many record types:
                #   "warcinfo"  → metadata about the crawl (skip)
                #   "request"   → the HTTP request sent (skip)
                #   "response"  → the HTTP response received (KEEP)
                #   "metadata"  → extra crawl metadata (skip)
                # We only want "response" records — they contain the HTML.
                if record.rec_type != "response":
                    continue

                # Within response records, filter to HTML only.
                # http_headers are the HTTP response headers (Content-Type, etc.)
                # rec_headers are the WARC-level headers (Record-ID, Date, etc.)
                # "or """ guards against get_header() returning None.
                content_type = record.http_headers.get_header("Content-Type") or ""
                if "html" not in content_type.lower():
                    continue

                # Read the response body and decode bytes → str.
                # errors="replace" substitutes undecodable bytes with ï�
                # instead of raising UnicodeDecodeError on malformed pages.
                html_content = record.content_stream().read().decode(
                    "utf-8", errors="replace"
                )

                all_rows.append({
                    # WARC-level headers: metadata about the crawl record
                    "warc_id":       record.rec_headers.get_header("WARC-Record-ID"),
                    "url":           record.rec_headers.get_header("WARC-Target-URI"),
                    "date":          record.rec_headers.get_header("WARC-Date"),
                    "html_content":  html_content,
                    # Store just the filename (not the full path) —
                    # Spark uses this as the partition folder name on disk.
                    "warc_filename": filename,
                })

    print(f"  Total records parsed: {len(all_rows)}")

    # Create a Spark DataFrame from the list of Python dicts.
    # Spark distributes these rows across partitions internally.
    df = spark.createDataFrame(all_rows, schema=schema)

    # Write as Parquet, partitioned by warc_filename.
    # This creates sub-folders like:
    #   PARQUET_DIR/warc_filename=CC-MAIN-...-00000.warc/part-00000.parquet
    #   PARQUET_DIR/warc_filename=CC-MAIN-...-00001.warc/part-00000.parquet
    # mode="overwrite" replaces any existing output (safe to re-run).
    df.write.partitionBy("warc_filename").parquet(PARQUET_DIR, mode="overwrite")
    print(f"✓ Stage 1 complete: {df.count()} records written to {PARQUET_DIR}")

#######################################
###!@1 END ANSWER STAGE 1
#######################################

## Stage 2: Data Ingestion & Domain Filtering ***(15 points)***
----


### Objective
Load raw WARC data from parquet files and filter out records from blacklisted domains.

### Input
- `input_path`: HDFS (Colab for this notebook) path to parquet files containing WARC records

### Processing Steps
1. **Construct Domain Blacklist**:
    - Recursively walk the `blacklist_root_dir`.
    - Identify files named `domains` or `urls`.
    - Read each file line-by-line, stripping whitespace.
    - Ignore lines starting with `#` (comments).
    - The aforementioned steps are guidelines. Ultimately we want the ouput in the format described below.
    - Output can look like - `{'altabu-db1.blogspot.hu',
 'biwaisms-putesdefoncees.blogspot.cz',
 'kamilla18.blogspot.jp',
 'batorsparadise.blogspot.pt', ...}`
2. Read parquet files from the output directory of the previous stage.
3. Extract domain from URL using regex.
4. Filter out records where domain and subdomains matches blacklist.


### Output Schema
| Column | Type | Description |
|--------|------|-------------|
| warc_id | string | Unique WARC record identifier |
| url | string | Source URL |
| date | string | Crawl timestamp |
| html_content | string | Raw HTML content |

### Expected Output Example (Stage 2)
```
root
|-- warc_id: string (nullable = true)
|-- url: string (nullable = true)
|-- date: string (nullable = true)
|-- html_content: string (nullable = true)

[Stage 109:>                                                        (0 + 1) / 1]
+--------------------+--------------------+--------------------+--------------------+
|             warc_id|                 url|                date|        html_content|
+--------------------+--------------------+--------------------+--------------------+
|<urn:uuid:ed46a75...|https://www.futur...|2025-12-04T20:34:05Z|<!DOCTYPE html><h...|
|<urn:uuid:9fb9dc9...|https://amotion.t...|2025-12-04T19:35:27Z|\n\n<!DOCTYPE htm...|
|<urn:uuid:98bada1...|https://www.fuzzy...|2025-12-04T20:30:52Z|<!DOCTYPE html>\n...|
|<urn:uuid:7149527...|https://amp.rtve....|2025-12-04T19:49:05Z|\n  <!DOCTYPE htm...|
|<urn:uuid:423fa66...|https://www.fwg-p...|2025-12-04T19:25:30Z|<!DOCTYPE html>\n...|
+--------------------+--------------------+--------------------+--------------------+
only showing top 5 rows

Schema Validation Passed!
✓ Stage 2 schema validation passed!

In [ ]:
#######################################
###!@1 START ANSWER STAGE 2
#######################################

def load_blacklist_categories():
    """
    Recursively walks BLACKLIST_DOMAINS_DIR and collects every domain/url
    entry from files named "domains" or "urls" into a Python set.
    Lines starting with "#" are comments and are ignored.
    Returns a set of domain strings for O(1) lookup during filtering.
    """
    print("Loading blacklist categories...")
    blacklisted_domains = set()

    # os.walk() recursively visits every subfolder under BLACKLIST_DOMAINS_DIR.
    # For each folder it yields:
    #   dirpath   → the current folder full path
    #   dirnames  → list of subfolders inside dirpath (unused here, hence _)
    #   filenames → list of files inside dirpath
    for dirpath, _, filenames in os.walk(BLACKLIST_DOMAINS_DIR):
        for filename in filenames:

            # Only process files literally named "domains" or "urls".
            # Anything else (README, .DS_Store, etc.) is ignored.
            if filename not in ("domains", "urls"):
                continue

            filepath = os.path.join(dirpath, filename)
            with open(filepath, "r", errors="replace") as f:
                for line in f:
                    line = line.strip()  # remove leading/trailing whitespace

                    # Skip empty lines and comment lines (starting with "#")
                    if not line or line.startswith("#"):
                        continue

                    blacklisted_domains.add(line)

    print(f"  → Loaded {len(blacklisted_domains)} blacklisted domains.")
    return blacklisted_domains


def stage_2_ingestion(input_path):
    """
    Loads WARC parquet files and removes records from blacklisted domains.
    Args:
        input_path: Path to the Parquet files written by Stage 1.
    Returns:
        DataFrame: Filtered dataframe — blacklisted domains removed.
                   Columns: warc_id, url, date, html_content
    """

    # ── Step 1: Build the blacklist ──────────────────────────────────────────
    blacklisted_domains = load_blacklist_categories()

    # Broadcast the Python set to all Spark workers.
    # Without broadcast: Spark re-serializes the entire set for EVERY row
    # processed by the UDF — catastrophically slow on large data.
    # With broadcast: each worker gets ONE efficient cached copy in memory.
    broadcast_blacklist = spark.sparkContext.broadcast(blacklisted_domains)


    # ── Step 2: Define the blacklist check UDF ───────────────────────────────
    def is_blacklisted(domain):
        """
        Returns True if domain exactly matches OR is a subdomain of
        any entry in the blacklist.

        Example:
            blacklist = {"blogspot.jp"}
            "blogspot.jp"           → True  (exact match)
            "kamilla18.blogspot.jp" → True  (subdomain match)
            "otherblogspot.jp"      → False (different domain)
        """
        if not domain:
            return False

        # .value accesses the broadcasted set on THIS worker node.
        bl = broadcast_blacklist.value

        # Exact match check (fastest path — O(1) set lookup)
        if domain in bl:
            return True

        # Subdomain check: progressively strip left labels and check.
        # e.g. "a.b.c.com" → check "b.c.com", then "c.com", then "com"
        parts = domain.split(".")
        for i in range(1, len(parts)):
            parent = ".".join(parts[i:])
            if parent in bl:
                return True

        return False

    # Register the Python function as a Spark UDF returning a boolean.
    # BooleanType because this UDF drives filtering (True = blacklisted = drop).
    is_blacklisted_udf = F.udf(is_blacklisted, BooleanType())


    # ── Step 3: Load Parquet from Stage 1 ────────────────────────────────────
    # Spark automatically discovers all partition sub-folders and
    # reconstructs the full schema from Parquet metadata. No schema needed.
    df = spark.read.parquet(input_path)


    # ── Step 4: Extract the hostname from the URL ─────────────────────────────
    # regexp_extract(column, pattern, group_index) extracts a regex capture group.
    #
    # Pattern: r"https?://([^/?\s]+)"
    #   https?      → matches "http" or "https"
    #   ://         → literal
    #   ([^/?\s]+)  → capture group 1: chars that are NOT /, ?, or whitespace
    #                  i.e. just the hostname
    #
    # "https://www.kamilla18.blogspot.jp/post" → "www.kamilla18.blogspot.jp"
    df = df.withColumn(
        "domain",
        F.regexp_extract(F.col("url"), r"https?://([^/?\s]+)", 1)
    )


    # ── Step 5: Filter out blacklisted domains ───────────────────────────────
    # ~ is Spark column NOT. We KEEP rows where is_blacklisted returns False.
    df_filtered = df.filter(~is_blacklisted_udf(F.col("domain")))


    # ── Step 6: Drop the temporary domain column ─────────────────────────────
    # "domain" was only used for filtering logic.
    # Required output schema: warc_id, url, date, html_content — no domain.
    output_df = df_filtered.drop("domain")

    return output_df

#######################################
###!@1 END ANSWER STAGE 2
#######################################

---
## Stage 3: Text Extraction ***(15 points)***
----

### Objective
Extract clean, readable text content from raw HTML using the Trafilatura library.

### Input
- DataFrame with `html_content` column (from Stage 2)

### Processing Steps (Guidelines)
1. Apply Trafilatura extraction.

**Specific instructions on HTML tags to include/exclude, images to include/exclude will be provided soon**

### Output Schema
| Column | Type | Description |
|--------|------|-------------|
| warc_id | string | Unique WARC record identifier |
| url | string | Source URL |
| date | string | Crawl timestamp |
| extracted_text | string | Clean text extracted from HTML |

### Expected Output Example (Stage 3)
```
root
 |-- warc_id: string (nullable = true)
 |-- url: string (nullable = true)
 |-- date: string (nullable = true)
 |-- extracted_text: string (nullable = true)

[Stage 117:>                                                        (0 + 1) / 1]
+--------------------+--------------------+--------------------+---------------------------+
|             warc_id|                 url|                date|             extracted_text|
+--------------------+--------------------+--------------------+---------------------------+
|<urn:uuid:51e562d...|https://www.viagr...|2025-12-04T19:37:35Z|产品分类\nProducts相关文...|
|<urn:uuid:9422649...|http://www.elolit...|2025-12-04T20:43:27Z|       Confinarse, encer...|
|<urn:uuid:e4642c3...|https://www.viapi...|2025-12-04T20:27:35Z|       Warenkorb ist lee...|
|<urn:uuid:85aa3aa...|http://www.empren...|2025-12-04T20:47:21Z|       ¿Qué es el delito...|
|<urn:uuid:7c4b999...|https://www.vibar...|2025-12-04T19:39:43Z|       317 Products\nΚαν...|
+--------------------+--------------------+--------------------+---------------------------+
only showing top 5 rows

Schema Validation Passed!
✓ Stage 3 schema validation passed!

```

In [ ]:
#######################################
###!@2 START ANSWER STAGE 3
#######################################

def stage_3_extraction(input_df):
    """
    Extracts clean readable text from raw HTML using Trafilatura.
    Rows where extraction fails (login walls, 404s, JS-only pages) are dropped.
    Args:
        input_df: DataFrame with "html_content" column (from Stage 2).
    Returns:
        DataFrame: "html_content" replaced by "extracted_text".
                   Columns: warc_id, url, date, extracted_text
    """

    # ── Step 1: Define the extraction function ───────────────────────────────
    # This plain Python function receives ONE html string per call.
    # Spark calls it row-by-row across all partitions on the workers.
    def extract_text(html):
        # Guard: if html is None/empty (edge case), return None immediately.
        # Never pass None into trafilatura — it raises a TypeError.
        if not html:
            return None

        # trafilatura.extract() internally:
        #   1. Parses the HTML tree (via lxml)
        #   2. Scores each block by text density, link ratio, tag semantics
        #   3. Extracts the highest-scoring content block (the "main article")
        #   4. Returns a clean string, OR None if no meaningful content found
        #      (e.g. pure navigation pages, error pages, login walls)
        #
        # NOTE: When your professor shares specific tag/image include/exclude
        # instructions, add them here as keyword arguments:
        #   trafilatura.extract(html, include_tables=True, include_images=False)
        return trafilatura.extract(html)


    # ── Step 2: Register as a Spark UDF returning a string ──────────────────
    # StringType() because this UDF transforms the column value (not filters).
    # Python None returned by extract_text maps to Spark null automatically.
    extract_udf = F.udf(extract_text, StringType())


    # ── Step 3: Apply the UDF to create the "extracted_text" column ──────────
    # withColumn(name, expr): adds a new column (or replaces if name exists).
    # Each Spark worker calls extract_text() on its local partition rows.
    df = input_df.withColumn(
        "extracted_text",
        extract_udf(F.col("html_content"))
    )


    # ── Step 4: Drop rows where extraction returned None ─────────────────────
    # isNotNull() is the Spark equivalent of Python "is not None".
    # Never use "!= None" on Spark columns — Spark has its own null semantics
    # and Python-style None comparisons silently produce wrong results.
    df = df.filter(F.col("extracted_text").isNotNull())


    # ── Step 5: Drop the raw HTML column ─────────────────────────────────────
    # The HTML has done its job. Dropping it frees significant memory/storage.
    # Required output schema: warc_id, url, date, extracted_text.
    output_df = df.drop("html_content")

    return output_df

#######################################
###!@2 END ANSWER STAGE 3
#######################################

---
## Stage 4: Language Identification ***(15 points)***
----

### Objective
Filter documents to retain only English content using FastText language identification.

### Input
- DataFrame with `extracted_text` column (from Stage 3)
- `threshold`: Minimum probability for English classification (default: 0.6). We will test your code with different thresholds.

### Processing Steps
1. Predict language for each document
2. Filter to retain only documents classified as English (`__label__en`) with probability ≥ threshold

### Output Schema
Same as input (records that pass the English filter)

### Expected Output Example (Stage 4)
```
root
 |-- warc_id: string (nullable = true)
 |-- url: string (nullable = true)
 |-- date: string (nullable = true)
 |-- extracted_text: string (nullable = true)

[Stage 124:>                                                        (0 + 1) / 1]
+--------------------+--------------------+--------------------+--------------------+
|             warc_id|                 url|                date|      extracted_text|
+--------------------+--------------------+--------------------+--------------------+
|<urn:uuid:d119f3d...|https://snapvrs.o...|2025-12-04T20:39:17Z|As solar panels d...|
|<urn:uuid:75d7f1f...|https://fresh-tri...|2025-12-04T20:30:35Z|Diving in Jordan,...|
|<urn:uuid:bb9ce09...|https://snippets....|2025-12-04T21:11:11Z|Are RV campers a ...|
|<urn:uuid:4ed2248...|https://frigidair...|2025-12-04T20:47:04Z|Frigidaire Dryer ...|
|<urn:uuid:b0b3105...|https://soap2day....|2025-12-04T20:50:59Z|Genre\nAction\nAd...|
+--------------------+--------------------+--------------------+--------------------+
only showing top 5 rows

Schema Validation Passed!
✓ Stage 4 schema validation passed!
```

In [ ]:
#######################################
###!@3 START ANSWER STAGE 4
#######################################
from pyspark import SparkFiles
import fasttext

# Module-level model cache — lives outside the UDF.
# First call on a worker: loads model from disk (~1-2 sec, done ONCE).
# Every subsequent call on the same worker: instant reuse.
# Without this, fasttext.load_model() would run for EVERY row — catastrophic.
_model = None

def get_model():
    """
    Lazy singleton loader for the FastText model.
    Loads the model from the Spark-distributed local copy on first call,
    then caches it in _model for all subsequent calls on this worker.
    """
    global _model
    # global is required: without it, Python treats _model as a new local
    # variable and the assignment is thrown away after the function returns.
    if _model is None:
        # SparkFiles.get(filename) returns the LOCAL disk path where
        # sc.addFile() copied the file on THIS worker.
        # Pass only the filename — NOT the full Drive path.
        # e.g. "/tmp/spark-abc123/userFiles/lid.176.bin"
        local_model_path = SparkFiles.get("lid.176.bin")
        _model = fasttext.load_model(local_model_path)
    return _model


def stage_4_lang_id(input_df, threshold=0.6):
    """
    Filters the DataFrame to retain only English-language documents.
    Args:
        input_df:  DataFrame with "extracted_text" column (from Stage 3).
        threshold: Minimum FastText confidence to accept as English (default 0.6).
                   The grader will test with different threshold values.
    Returns:
        DataFrame: English-only rows. Schema identical to input.
    """

    # ── Step 1: Distribute the model binary to all workers ───────────────────
    #
    # PROBLEM: The FastText model is a ~900MB C++-backed binary.
    #   It cannot be Python-pickled and broadcast() like a dict or set.
    #   Workers need a LOCAL copy of the file on their own disk to load it.
    #
    # SOLUTION — sc.addFile() + SparkFiles.get():
    #   sc.addFile(path)          → Driver tells Spark to copy this file
    #                               to every worker local disk before tasks run.
    #   SparkFiles.get(filename)  → Inside UDF, returns the local disk path
    #                               of the copied file on THIS worker.
    #
    # FLOW:
    #   Driver:  sc.addFile("/content/drive/.../lid.176.bin")
    #                │
    #                └─► Worker local disk: /tmp/spark-xxx/userFiles/lid.176.bin
    #                                               │
    #                                  SparkFiles.get("lid.176.bin")
    #                                               │
    #                                  fasttext.load_model(local_path)
    #
    # NOTE: In Colab local[*] mode, all workers are threads on the same machine
    # so the Drive path would technically work too — but addFile is the correct,
    # generalizable pattern and matches the starter code imports.
    spark.sparkContext.addFile(FASTTEXT_MODEL_BIN)


    # ── Step 2: Define the language detection function ────────────────────────
    def is_english(text):
        # Guard: null or empty text cannot be classified → discard.
        if not text or not text.strip():
            return False

        model = get_model()  # instant after first load on this worker

        # CRITICAL: FastText treats each LINE as a separate document.
        # Newlines inside the text confuse prediction — it only classifies
        # the first line and silently ignores the rest.
        # Replace all newlines/carriage returns with spaces before predicting.
        clean_text = text.replace("
", " ").replace("", " ").strip()

        # model.predict() returns a tuple of two elements:
        #   predictions[0] → tuple of label strings  e.g. ("__label__en",)
        #   predictions[1] → numpy array of floats   e.g. array([0.9999])
        # By default, it returns only the TOP 1 prediction.
        predictions = model.predict(clean_text)

        top_label = predictions[0][0]        # e.g. "__label__en"
        top_prob  = float(predictions[1][0]) # e.g. 0.9999

        # Keep the document ONLY IF:
        #   1. FastText classified it as English (__label__en)
        #   2. Confidence is at or above the threshold (captured via closure)
        return (top_label == "__label__en") and (top_prob >= threshold)


    # ── Step 3: Register as a Spark UDF returning Boolean ────────────────────
    # BooleanType() because this UDF drives filtering (True = keep row),
    # unlike Stage 3 UDF which returned StringType() for value transformation.
    is_english_udf = F.udf(is_english, BooleanType())


    # ── Step 4: Filter — keep only English documents ─────────────────────────
    # Output schema is IDENTICAL to input — we only remove non-English rows,
    # no columns are added or dropped.
    output_df = input_df.filter(is_english_udf(F.col("extracted_text")))

    return output_df

#######################################
###!@3 END ANSWER STAGE 4
#######################################

---
## Stage 5: Near-Duplicate Deduplication ***(30 points)***
----

### Objective
Remove near-duplicate documents using MinHash Locality-Sensitive Hashing (LSH).

### Input
- DataFrame with `extracted_text` column (from Stage 4)

### Algorithm
1. **Character 5-grams**: Convert text to character-level 5-grams, hash each to a vocabulary index
2. **MinHash Signatures**: Generate 24 MinHash signatures using Spark MLlib's MinHashLSH
3. **Banding**: Split signatures into 8 bands of 3 hashes each for similarity detection
4. **Clustering**: Find similar document pairs with Jaccard distance ≤ 0.5
5. **Filtering**: Keep only the longest document from each cluster

### Configuration
| Parameter | Value | Description |
|-----------|-------|-------------|
| NGRAM_SIZE | 5 | Character n-gram length |
| NUM_HASHES | 24 | Total MinHash signatures |
| NUM_BANDS | 8 | Number of LSH bands |
| VOCAB_SIZE | 2^18 | Hash space size (262,144) |

### Output Schema
Same as input (deduplicated records)

### Expected Output Example (Stage 5)
```
root
 |-- warc_id: string (nullable = true)
 |-- url: string (nullable = true)
 |-- date: string (nullable = true)
 |-- extracted_text: string (nullable = true)

[Stage 155:======================================================>(96 + 1) / 97]
+--------------------+--------------------+--------------------+--------------------+
|             warc_id|                 url|                date|      extracted_text|
+--------------------+--------------------+--------------------+--------------------+
|         warc_id_102|http://example.co...|2025-12-04T21:02:54Z|Machine learning ...|
|         warc_id_103|http://example.co...|2025-12-04T21:02:55Z|Python is a high-...|
|<urn:uuid:caa5d51...|https://www.valte...|2025-12-04T20:38:04Z|Agentforce: The G...|
|<urn:uuid:15335f8...|https://www.vande...|2025-12-04T20:39:07Z|This Cookie Polic...|
|<urn:uuid:b5590de...|http://www.elks.o...|2025-12-04T20:06:13Z|Join the Elks!\nL...|
+--------------------+--------------------+--------------------+--------------------+
only showing top 5 rows

Schema Validation Passed!
✓ Stage 5 schema validation passed!
```

In [ ]:
#######################################
###!@4 START ANSWER STAGE 5
#######################################
from pyspark.ml.feature import MinHashLSH
from pyspark.ml.linalg import Vectors, VectorUDT
from pyspark.sql import Window
import hashlib

def stage_5_deduplication(input_df):
    """
    Args:
        input_df: English text dataframe.
    Returns:
        DataFrame: Deduplicated dataframe.
    """

    return output_df

#######################################
###!@4 END ANSWER STAGE 5
#######################################

---
## Stage 6: Tokenization ***(20 points)***
----

### Objective
Convert cleaned text into fixed-length token sequences suitable for LLM training.

### Input
- DataFrame with text column (`extracted_text`, `cleaned_text`, or `text`)

### Tokenization Backends
| Backend | Description |
|---------|-------------|
| `regex` | Word-level tokenization using `\b\w+\b` pattern, hash-based IDs (default) |
| `whitespace` | Simple space-split tokenization, hash-based IDs |
| `sentencepiece` | Subword tokenization (requires model_path) |

### Parameters
| Parameter | Default | Description |
|-----------|---------|-------------|
| max_length | 512 | Maximum sequence length |
| vocab_size | 50000 | Hash space for token IDs |
| pad_id | 0 | Padding token ID |

### Output Schema
| Column | Type | Description |
|--------|------|-------------|
| warc_id | string | Unique WARC record identifier |
| tokens | array<int> | Token IDs (padded to max_length) |
| attention_mask | array<int> | 1 for real tokens, 0 for padding |

### Expected Output Example (Stage 6)
```
root
 |-- warc_id: string (nullable = true)
 |-- tokens: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- attention_mask: array (nullable = true)
 |    |-- element: integer (containsNull = true)

+--------------------+--------------------+--------------------+
|             warc_id|              tokens|      attention_mask|
+--------------------+--------------------+--------------------+
|         warc_id_102|[8969, 22291, 342...|[1, 1, 1, 1, 1, 1...|
|<urn:uuid:11e4b22...|[26423, 23563, 48...|[1, 1, 1, 1, 1, 1...|
|<urn:uuid:b929105...|[7260, 111, 9755,...|[1, 1, 1, 1, 1, 1...|
|<urn:uuid:e26e300...|[37479, 10280, 10...|[1, 1, 1, 1, 1, 1...|
|<urn:uuid:d119f3d...|[11201, 47384, 31...|[1, 1, 1, 1, 1, 1...|
+--------------------+--------------------+--------------------+
only showing top 5 rows

Schema Validation Passed!
✓ Stage 6 schema validation passed!
```

In [ ]:
#######################################
###!@4 START ANSWER STAGE 6
#######################################

def stage_6_tokenization(input_df):
    """
    Tokenize text. Stdlib-only (no external libs) or sentencepiece.
    Args:
        input_df: Cleaned text dataframe.
        encoding_name: Unused for regex/whitespace.
        max_length: Max sequence length (default 512).
        backend: "regex" (default, stdlib only), "whitespace", or "sentencepiece".
          - regex: words via re (\\b\\w+\\b), hash-based ids.
          - whitespace: split on spaces, hash-based ids.
          - sentencepiece: subword, requires model_path.
        model_path: Required when backend="sentencepiece".
        vocab_size: For regex/whitespace, hash space size (default 50000).
    Returns:
        DataFrame: Columns ['warc_id', 'tokens', 'attention_mask']
    """

    return output_df

#######################################
###!@5 END ANSWER STAGE 6
#######################################

In [ ]:
# EXECUTE PIPELINE
# This cell checks the flow from start to finish.

print(">>> Starting Pipeline Execution...")

# 1. Convert the warc files to parquet
stage_1_warc_to_parquet()

# 2. Parse and url filtering
df_s2 = stage_2_ingestion()
validate_stage_schema(df_s2, 1)

# 3. Text extraction
df_s3 = stage_3_extraction(df_s2)
validate_stage_schema(df_s3, 2)

# 4. Language Identification
df_s4 = stage_4_lang_id(df_s3)
validate_stage_schema(df_s4, 3)

# 5. Deduplication
df_s5 = stage_5_deduplication(df_s4)
validate_stage_schema(df_s5, 4)

# 6. Tokenization
df_final = stage_6_tokenization(df_s5)